In [0]:
import sys, importlib, time, datetime as dt
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import src.oil_http as oh
import src.eia_loader as eia
import src.jodi_loader as jodi
import src.portwatch_loader as pw
import src.cot_loader as cot
import src.macro_loader as macro
import src.noaa_loader as noaa
import src.gdelt_loader as gd
for m in (oh, eia, jodi, pw, cot, macro, noaa, gd):
    importlib.reload(m)

EIA_KEY = dbutils.secrets.get("oil-shock", "eia-api-key")
FRED_KEY = dbutils.secrets.get("oil-shock", "fred-api-key")

dubai_now = dt.datetime.utcnow() + dt.timedelta(hours=4)
is_monday_morning = dubai_now.weekday() == 0 and dubai_now.hour < 12

steps = [
    ("EIA", lambda: eia.run(EIA_KEY, mode="daily",
                            codes=[s["code"] for s in eia.SERIES if not s["code"].startswith("RCLC")])),
    ("FRED", lambda: macro.run_fred(FRED_KEY, mode="recent")),
    ("OECD", lambda: macro.run_oecd(mode="recent")),
    ("PortWatch", lambda: pw.run(mode="recent")),
    ("CFTC", lambda: cot.run_cftc(mode="recent")),
    ("ICE", lambda: cot.run_ice(mode="recent")),
    ("NOAA active", lambda: noaa.run_active()),
    ("NOAA history", lambda: noaa.run_hurdat()),
    ("GDELT", lambda: gd.run_recent(max_hours=36)),
]
if is_monday_morning:
    steps.append(("JODI", lambda: jodi.run(mode="recent")))

results, failures = [], []
for name, fn in steps:
    t0 = time.time()
    try:
        out = fn()
        if isinstance(out, list):
            detail = f"{len(out)} items, {sum(x[1] for x in out if isinstance(x[1], (int, float))):,} rows"
        else:
            detail = str(out)[:120]
        results.append((name, "OK", round(time.time() - t0), detail))
    except Exception as e:
        failures.append(name)
        results.append((name, "FAILED", round(time.time() - t0), oh.redact(str(e))[:200]))

print(f"Run at {dubai_now:%Y-%m-%d %H:%M} Dubai | JODI included: {is_monday_morning}")
print(f"{'Step':<14} {'Result':<8} {'Secs':>5}  Detail")
print("-" * 110)
for name, status, secs, detail in results:
    print(f"{name:<14} {status:<8} {secs:>5}  {detail}")
print("-" * 110)

if failures:
    raise RuntimeError(f"Ingest finished with failures: {failures}")
print("All steps OK")